In [ ]:
# ============================================================================
#  CNN activity classifier - 5-fold subject-wise cross-validation
# ----------------------------------------------------------------------------
#  Data:  balanced_folds/fold_<i>/  built by data_processing.ipynb
#         X_* (N, 128, 6) float32 - 4 s segments, 32 Hz, acc x/y/z + gyro x/y/z
#         y_* (N,) int8            - class 0-6
#  Runs:  five independent trainings. Each model sees only its own fold's
#         train users, early-stops on its own val users, and predicts its own
#         test users. The five test sets are disjoint and cover all 56 users.
#
#  NOTE on torch/numpy: torch 2.2.0 here is compiled against NumPy 1.x and the
#  installed NumPy is 2.4.6, so torch.from_numpy() and tensor.numpy() both raise
#  "Numpy is not available". Everything below therefore crosses the boundary
#  through the buffer protocol (torch.frombuffer / .tolist()), which does not
#  touch the numpy C-API. Upgrading torch would also fix it, but the CUDA 12.1
#  build matches this driver (525.89.02) and a newer build may not.
# ============================================================================
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn

DATA_DIR    = "balanced_folds"
N_FOLDS     = 5
N_CLASSES   = 7
SEG_LEN     = 128
N_CHANNELS  = 6

BATCH       = 512
EVAL_BATCH  = 4096
MAX_EPOCHS  = 20
LR          = 1e-3
WEIGHT_DECAY= 1e-4
PATIENCE     = 4          # epochs without val macro-F1 improvement
SEED        = 42

CLASS_NAMES = ["Lying down", "Sitting", "Walking", "Running",
               "Bicycling", "Standing in place", "Standing and moving"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"torch {torch.__version__} | numpy {np.__version__}")

_TORCH_DTYPE = {"float32": torch.float32, "float64": torch.float64,
                "int8": torch.int8, "int16": torch.int16,
                "int32": torch.int32, "int64": torch.int64, "bool": torch.bool}


def to_torch(a, device=None):
    """numpy array -> torch tensor without using the (broken) numpy bridge."""
    a = np.ascontiguousarray(a)
    t = torch.frombuffer(memoryview(a.reshape(-1)),
                         dtype=_TORCH_DTYPE[a.dtype.name]).reshape(a.shape)
    return t.to(device) if device is not None else t.clone()


def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

In [ ]:
# ---------------------------------------------------------------------------
#  Metrics, computed from a confusion matrix
# ---------------------------------------------------------------------------
#  Everything we need - accuracy, macro-F1, balanced accuracy, Cohen's kappa -
#  is a function of the 7x7 confusion matrix, so predictions never have to be
#  converted to numpy (which the torch/numpy mismatch would block anyway).
#  Only 49 integers cross back from the GPU per evaluation.
# ---------------------------------------------------------------------------

def confusion(y_true, y_pred, n=N_CLASSES):
    """7x7 confusion matrix on-device. Rows = true class, cols = predicted."""
    k = (y_true.long() * n + y_pred.long())
    return torch.bincount(k, minlength=n * n).reshape(n, n)


def metrics_from_cm(cm):
    """accuracy, macro-F1, balanced accuracy, Cohen's kappa + per-class detail."""
    cm = np.asarray(cm.tolist(), dtype=np.float64)      # 49 numbers, no bridge
    total = cm.sum()
    if total == 0:
        return {}
    tp = np.diag(cm)
    row = cm.sum(1)                                      # support per true class
    col = cm.sum(0)                                      # predictions per class

    accuracy = tp.sum() / total

    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.where(row > 0, tp / np.maximum(row, 1), 0.0)
        precision = np.where(col > 0, tp / np.maximum(col, 1), 0.0)
        denom = precision + recall
        f1 = np.where(denom > 0, 2 * precision * recall / np.maximum(denom, 1e-12), 0.0)

    # macro averages ignore classes absent from the ground truth of this split
    present = row > 0
    macro_f1 = f1[present].mean()
    balanced_accuracy = recall[present].mean()

    p_e = float((row * col).sum()) / (total * total)     # chance agreement
    kappa = (accuracy - p_e) / (1 - p_e) if p_e < 1 else 0.0

    return {"accuracy": float(accuracy), "macro_f1": float(macro_f1),
            "balanced_accuracy": float(balanced_accuracy), "kappa": float(kappa),
            "per_class": {"precision": precision, "recall": recall, "f1": f1,
                          "support": row.astype(np.int64)},
            "cm": cm.astype(np.int64)}


def print_per_class(m, title="per-class"):
    pc = m["per_class"]
    print(f"\n  {title}")
    print(f"    {'idx':>3}  {'class':22s} {'precision':>10s} {'recall':>8s} "
          f"{'f1':>8s} {'support':>10s}")
    for k in range(N_CLASSES):
        print(f"    {k:>3}  {CLASS_NAMES[k]:22s} {pc['precision'][k]:10.3f} "
              f"{pc['recall'][k]:8.3f} {pc['f1'][k]:8.3f} {pc['support'][k]:10,}")

In [ ]:
# ---------------------------------------------------------------------------
#  Loading a fold onto the GPU
# ---------------------------------------------------------------------------
#  Each fold is ~2.6 GB across train/val/test, which fits comfortably on a 20 GB
#  card, so the whole fold is resident and batching is plain tensor indexing -
#  no DataLoader, no per-batch host->device copies.
#
#  Normalisation uses THIS fold's norm_mean / norm_std, which were fitted on its
#  real (non-augmented) training segments only. Using another fold's statistics,
#  or statistics computed over all data, would leak test users into training.
# ---------------------------------------------------------------------------

def load_fold(i, device=DEVICE):
    d = f"{DATA_DIR}/fold_{i}"
    mean = to_torch(np.load(f"{d}/norm_mean.npy"), device)      # (6,)
    std = to_torch(np.load(f"{d}/norm_std.npy"), device)

    out = {}
    for part in ("train", "val", "test"):
        X = np.load(f"{d}/X_{part}.npy")                        # (N, 128, 6)
        y = np.load(f"{d}/y_{part}.npy")
        Xt = to_torch(X, device)
        del X
        Xt = (Xt - mean) / std                                  # broadcast over channels
        Xt = Xt.permute(0, 2, 1).contiguous()                   # -> (N, 6, 128) for Conv1d
        out[part] = (Xt, to_torch(y, device).long())
        del y

    with open(f"{d}/class_weights.json") as fh:
        out["class_weights"] = to_torch(
            np.asarray(json.load(fh)["class_weights"], dtype="float32"), device)
    out["users"] = json.load(open(f"{d}/report.json"))["users"]
    return out


def describe_fold(f, i):
    tr, va, te = f["train"][0], f["val"][0], f["test"][0]
    used = sum(t.numel() * t.element_size() for t in (tr, va, te)) / 1e9
    print(f"  fold {i}: train {tuple(tr.shape)}  val {tuple(va.shape)}  "
          f"test {tuple(te.shape)}  [{used:.2f} GB on {DEVICE}]")
    print(f"          users  train {len(f['users']['train'])}  "
          f"val {len(f['users']['val'])}  test {len(f['users']['test'])}")

In [ ]:
# ---------------------------------------------------------------------------
#  The model: a two-scale 1D CNN
# ---------------------------------------------------------------------------
#  Two parallel first-layer branches read the same window at different time
#  scales, then their features are concatenated:
#     k=5  (~0.16 s) - fast transients: a footfall, a jolt
#     k=21 (~0.66 s) - a whole gait or pedal cycle
#  Forcing one kernel size to serve both is the usual reason a single-scale CNN
#  confuses Walking with Running.
#
#  Global average pooling rather than flatten: the phase of a gait cycle is not
#  aligned across segments, so the summary must be translation-invariant, and it
#  keeps the head small enough not to memorise the augmented rare classes.
# ---------------------------------------------------------------------------

class HARCNN(nn.Module):
    def __init__(self, in_ch=N_CHANNELS, n_classes=N_CLASSES, p_drop=0.4):
        super().__init__()
        self.fast = nn.Sequential(
            nn.Conv1d(in_ch, 48, kernel_size=5, padding=2),
            nn.BatchNorm1d(48), nn.ReLU(inplace=True))
        self.slow = nn.Sequential(
            nn.Conv1d(in_ch, 48, kernel_size=21, padding=10),
            nn.BatchNorm1d(48), nn.ReLU(inplace=True))
        self.body = nn.Sequential(
            nn.MaxPool1d(2),                                   # 128 -> 64
            nn.Conv1d(96, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2),                                   # 64 -> 32
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1))                           # -> (B, 128, 1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_drop),
            nn.Linear(128, 128), nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(128, n_classes))

    def forward(self, x):                                      # x: (B, 6, 128)
        x = torch.cat([self.fast(x), self.slow(x)], dim=1)     # (B, 96, 128)
        return self.head(self.body(x))


m = HARCNN()
print(f"HARCNN: {sum(p.numel() for p in m.parameters()):,} parameters")
print(f"  forward check: {tuple(m(torch.zeros(2, N_CHANNELS, SEG_LEN)).shape)} "
      f"(expect (2, {N_CLASSES}))")
del m

In [ ]:
# ---------------------------------------------------------------------------
#  Train / evaluate one fold
# ---------------------------------------------------------------------------
#  Early stopping watches validation MACRO-F1, not loss or accuracy. Validation
#  is deliberately left at the natural class distribution (~44% Sitting, 0.4%
#  Running), so loss and accuracy are both dominated by the majority classes and
#  would happily select a model that never predicts Running at all.
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, X, y, batch=EVAL_BATCH, return_preds=False):
    model.eval()
    cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=X.device)
    preds = [] if return_preds else None
    for s in range(0, len(X), batch):
        p = model(X[s:s + batch]).argmax(1)
        cm += confusion(y[s:s + batch], p)
        if return_preds:
            preds.append(p)
    m = metrics_from_cm(cm)
    return (m, torch.cat(preds)) if return_preds else (m, None)


def train_fold(i, verbose=True):
    set_seed(SEED + i)
    f = load_fold(i)
    if verbose:
        describe_fold(f, i)
    Xtr, ytr = f["train"]
    Xva, yva = f["val"]
    Xte, yte = f["test"]

    model = HARCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                       factor=0.5, patience=1)
    lossf = nn.CrossEntropyLoss(weight=f["class_weights"])

    best = {"macro_f1": -1.0}
    best_state, best_epoch, since = None, -1, 0
    n = len(Xtr)
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        perm = torch.randperm(n, device=DEVICE)
        run_loss = 0.0
        for s in range(0, n, BATCH):
            idx = perm[s:s + BATCH]
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(Xtr[idx]), ytr[idx])
            loss.backward()
            opt.step()
            run_loss += loss.item() * len(idx)
        vm, _ = evaluate(model, Xva, yva)
        sched.step(vm["macro_f1"])
        history.append({"epoch": epoch, "loss": run_loss / n, **{k: vm[k] for k in
                        ("accuracy", "macro_f1", "balanced_accuracy", "kappa")}})
        if verbose:
            print(f"    epoch {epoch:2d}  loss {run_loss / n:.4f}  "
                  f"val macro-F1 {vm['macro_f1']:.4f}  acc {vm['accuracy']:.4f}  "
                  f"bal-acc {vm['balanced_accuracy']:.4f}  ({time.time() - t0:.0f}s)")

        if vm["macro_f1"] > best["macro_f1"] + 1e-5:
            best, best_epoch, since = vm, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= PATIENCE:
                if verbose:
                    print(f"    early stop (no val improvement for {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)                 # restore the best epoch
    test_m, test_p = evaluate(model, Xte, yte, return_preds=True)
    if verbose:
        print(f"    best epoch {best_epoch} (val macro-F1 {best['macro_f1']:.4f})"
              f"  ->  TEST macro-F1 {test_m['macro_f1']:.4f}  "
              f"acc {test_m['accuracy']:.4f}  bal-acc {test_m['balanced_accuracy']:.4f}  "
              f"kappa {test_m['kappa']:.4f}")

    out = {"fold": i, "best_epoch": best_epoch, "history": history,
           "val": {k: best[k] for k in ("accuracy", "macro_f1",
                                        "balanced_accuracy", "kappa")},
           "test": test_m,
           "y_true": yte.cpu().tolist(), "y_pred": test_p.cpu().tolist()}
    del f, Xtr, ytr, Xva, yva, Xte, yte, model, best_state
    torch.cuda.empty_cache()
    return out

In [ ]:
# ---------------------------------------------------------------------------
#  Run all five folds
# ---------------------------------------------------------------------------
results = []
t_start = time.time()
for i in range(N_FOLDS):
    print(f"\n{'=' * 72}\nFOLD {i}\n{'=' * 72}")
    results.append(train_fold(i))
print(f"\nall folds complete in {(time.time() - t_start) / 60:.1f} min")

os.makedirs("cnn_results", exist_ok=True)
json.dump([{k: v for k, v in r.items() if k not in ("test", "y_true", "y_pred")}
           | {"test": {k: v for k, v in r["test"].items()
                       if k not in ("per_class", "cm")}}
           for r in results], open("cnn_results/folds.json", "w"), indent=1)
json.dump({str(r["fold"]): {"y_true": r["y_true"], "y_pred": r["y_pred"]}
           for r in results}, open("cnn_results/predictions.json", "w"))
print("saved -> cnn_results/folds.json, cnn_results/predictions.json")

In [ ]:
# ---------------------------------------------------------------------------
#  Per-fold results, and mean +/- standard deviation across the five folds
# ---------------------------------------------------------------------------
METRICS = [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"),
           ("balanced_accuracy", "Balanced accuracy"), ("kappa", "Cohen's kappa")]

vals = {k: np.array([r["test"][k] for r in results]) for k, _ in METRICS}

print("TEST metrics per fold\n")
print(f"  {'fold':>4} {'n_test':>10} {'epochs':>7}  "
      + "".join(f"{lab:>19s}" for _, lab in METRICS))
for r in results:
    print(f"  {r['fold']:>4} {int(r['test']['per_class']['support'].sum()):>10,} "
          f"{r['best_epoch']:>7}  "
          + "".join(f"{r['test'][k]:19.4f}" for k, _ in METRICS))

print(f"\n  {'mean':>4} {'':>10} {'':>7}  "
      + "".join(f"{vals[k].mean():19.4f}" for k, _ in METRICS))
print(f"  {'std':>4} {'':>10} {'':>7}  "
      + "".join(f"{vals[k].std(ddof=1):19.4f}" for k, _ in METRICS))

print("\n" + "-" * 72)
print("SUMMARY  (mean +/- std over 5 subject-wise folds)\n")
for k, lab in METRICS:
    print(f"  {lab:20s} {vals[k].mean():.4f}  +/-  {vals[k].std(ddof=1):.4f}"
          f"     [min {vals[k].min():.4f}, max {vals[k].max():.4f}]")

# baselines on the pooled test set, for reference
print("\n  for reference, on this data:")
print("    always-predict-Sitting        accuracy 0.441   macro-F1 0.087   kappa 0.000")
print("    only the 2 majority classes   accuracy 0.787   macro-F1 0.258")

In [ ]:
# ---------------------------------------------------------------------------
#  Pooled result across all five folds
# ---------------------------------------------------------------------------
#  Every user is tested exactly once, so concatenating the five prediction sets
#  gives ONE estimate over all 56 users. This is preferred to averaging the five
#  fold scores, because the test folds differ in size by nearly 3x (133k to
#  371k segments) and equal weighting would misrepresent that.
# ---------------------------------------------------------------------------
yt = torch.tensor([v for r in results for v in r["y_true"]])
yp = torch.tensor([v for r in results for v in r["y_pred"]])
pooled = metrics_from_cm(confusion(yt, yp))

print(f"pooled over {len(yt):,} test segments from all 56 users\n")
for k, lab in METRICS:
    print(f"  {lab:20s} {pooled[k]:.4f}")
print_per_class(pooled, "per-class (pooled)")

print("\n  confusion matrix, row-normalised (rows = true class)\n")
cm = pooled["cm"].astype(np.float64)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
print("    " + " " * 24 + "".join(f"{i:>8d}" for i in range(N_CLASSES)))
for i in range(N_CLASSES):
    print(f"    {i} {CLASS_NAMES[i]:22s}"
          + "".join(f"{cmn[i, j]:8.3f}" for j in range(N_CLASSES)))

json.dump({k: pooled[k] for k, _ in METRICS}
          | {"per_class_f1": pooled["per_class"]["f1"].tolist(),
             "per_class_recall": pooled["per_class"]["recall"].tolist(),
             "per_class_support": pooled["per_class"]["support"].tolist(),
             "confusion_matrix": pooled["cm"].tolist(),
             "class_names": CLASS_NAMES},
          open("cnn_results/pooled.json", "w"), indent=1)
print("\nsaved -> cnn_results/pooled.json")

In [ ]:
# ---------------------------------------------------------------------------
#  TABLE 1 - per-class metrics, pooled over all five folds
# ---------------------------------------------------------------------------
#  Every user is tested exactly once across the five folds, so concatenating the
#  predictions gives one estimate over all 56 users.
#
#  "accuracy", "macro F1" and "balanced accuracy" are aggregate metrics over all
#  classes - there is no macro-F1 "for Sitting". The per-class equivalents are
#  recall, precision and F1. Two one-vs-rest columns are included as well:
#     bal-acc  = (recall + specificity) / 2, treating the class as one-vs-rest
#     OvR acc  = (TP + TN) / total for that class against all others
#  OvR accuracy is reported because it is often asked for, but it is misleading
#  for rare classes: Running scores 0.96 there simply because 99.6% of segments
#  are not Running. Read F1 and recall instead.
# ---------------------------------------------------------------------------

def fold_predictions():
    """(y_true, y_pred) per fold - from memory if trained, else from disk."""
    if "results" in globals():
        return [(np.asarray(r["y_true"]), np.asarray(r["y_pred"])) for r in results]
    P = json.load(open("cnn_results/predictions.json"))
    return [(np.asarray(P[str(i)]["y_true"]), np.asarray(P[str(i)]["y_pred"]))
            for i in range(N_FOLDS)]


def confusion_np(y_true, y_pred, n=N_CLASSES):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)          # rows = true, cols = predicted
    return cm


folds = fold_predictions()
cm = sum(confusion_np(yt, yp) for yt, yp in folds)

tp = np.diag(cm).astype(np.float64)
support = cm.sum(1)                              # true instances per class
predicted = cm.sum(0)                            # predictions made per class
total = cm.sum()

recall = tp / np.maximum(support, 1)
precision = np.where(predicted > 0, tp / np.maximum(predicted, 1), 0.0)
f1 = np.where(precision + recall > 0,
              2 * precision * recall / np.maximum(precision + recall, 1e-12), 0.0)
specificity = (total - support - (predicted - tp)) / np.maximum(total - support, 1)
bal_acc = (recall + specificity) / 2             # one-vs-rest balanced accuracy
ovr_acc = (total - (support - tp) - (predicted - tp)) / total

print(f"TABLE 1 - per-class metrics, pooled over {N_FOLDS} folds "
      f"({total:,} test segments)\n")
print(f"{'idx':>3}  {'activity':22s} {'recall':>8s} {'precision':>10s} {'F1':>8s} "
      f"{'bal-acc':>9s} {'OvR acc':>9s} {'support':>11s}")
print("-" * 86)
for k in range(N_CLASSES):
    print(f"{k:>3}  {CLASS_NAMES[k]:22s} {recall[k]:8.3f} {precision[k]:10.3f} "
          f"{f1[k]:8.3f} {bal_acc[k]:9.3f} {ovr_acc[k]:9.3f} {support[k]:11,}")
print("-" * 86)
print(f"{'':>3}  {'macro average':22s} {recall.mean():8.3f} {precision.mean():10.3f} "
      f"{f1.mean():8.3f} {bal_acc.mean():9.3f}")
print(f"{'':>3}  {'overall accuracy':22s} {'':>8s} {'':>10s} {'':>8s} {'':>9s} "
      f"{tp.sum() / total:9.3f} {total:11,}")
print("\n  note: OvR accuracy flatters rare classes - Running reads 0.96 there "
      "while its F1 is 0.03,\n        because 99.6% of segments are not Running. "
      "F1 and recall are the honest columns.")

In [ ]:
# ---------------------------------------------------------------------------
#  TABLE 2 - per-class score for each fold separately
# ---------------------------------------------------------------------------
#  The spread across folds matters as much as the mean here. Only ~5 test users
#  per fold have Running or Bicycling, so which individuals land in which fold
#  moves these numbers more than the model does. A large std on a rare class is
#  a statement about the fold assignment, not about the classifier.
# ---------------------------------------------------------------------------

TABLE2_METRIC = "f1"          # "f1", "recall" or "precision"

per_fold = np.full((N_FOLDS, N_CLASSES), np.nan)
for i, (yt, yp) in enumerate(folds):
    c = confusion_np(yt, yp)
    d = np.diag(c).astype(np.float64)
    sup, pred = c.sum(1), c.sum(0)
    rec = np.where(sup > 0, d / np.maximum(sup, 1), np.nan)
    pre = np.where(pred > 0, d / np.maximum(pred, 1), 0.0)
    if TABLE2_METRIC == "recall":
        per_fold[i] = rec
    elif TABLE2_METRIC == "precision":
        per_fold[i] = np.where(sup > 0, pre, np.nan)
    else:
        r0 = np.nan_to_num(rec)
        per_fold[i] = np.where(sup > 0,
                               np.where(pre + r0 > 0,
                                        2 * pre * r0 / np.maximum(pre + r0, 1e-12), 0.0),
                               np.nan)

print(f"TABLE 2 - per-class {TABLE2_METRIC.upper()} across the {N_FOLDS} folds\n")
print(f"{'idx':>3}  {'activity':22s}"
      + "".join(f"{'fold ' + str(i):>9s}" for i in range(N_FOLDS))
      + f"{'mean':>9s}{'std':>8s}")
print("-" * 87)
for k in range(N_CLASSES):
    row = per_fold[:, k]
    print(f"{k:>3}  {CLASS_NAMES[k]:22s}"
          + "".join(f"{per_fold[i, k]:9.3f}" for i in range(N_FOLDS))
          + f"{np.nanmean(row):9.3f}{np.nanstd(row, ddof=1):8.3f}")
print("-" * 87)
print(f"{'':>3}  {'macro (mean of rows)':22s}"
      + "".join(f"{np.nanmean(per_fold[i]):9.3f}" for i in range(N_FOLDS))
      + f"{np.nanmean(per_fold):9.3f}")

worst = int(np.nanargmax(np.nanstd(per_fold, axis=0, ddof=1)))
lo, hi = np.nanmin(per_fold[:, worst]), np.nanmax(per_fold[:, worst])
print(f"\n  most fold-dependent class: {CLASS_NAMES[worst]} "
      f"({TABLE2_METRIC} ranges {lo:.3f} to {hi:.3f} across folds)")